In [1]:
library(Rcpp)
library(progress)
library(RcppEigen)
library(RcppDist)
library(RcppArmadillo)
library(mvtnorm)
library(dbarts)
sourceCpp("FirstModel.cpp")

Warning message:
"package 'Rcpp' was built under R version 4.3.3"
Warning message:
"package 'progress' was built under R version 4.3.3"
Warning message:
"package 'RcppEigen' was built under R version 4.3.3"
Warning message:
"package 'RcppDist' was built under R version 4.3.3"
Registered S3 methods overwritten by 'RcppArmadillo':
  method               from     
  predict.fastLm       RcppEigen
  print.fastLm         RcppEigen
  summary.fastLm       RcppEigen
  print.summary.fastLm RcppEigen


Attaching package: 'RcppArmadillo'


The following objects are masked from 'package:RcppEigen':

    fastLm, fastLmPure


Warning message:
"package 'mvtnorm' was built under R version 4.3.3"


# DGP_3

In [4]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
results_matrix <- matrix(NA, nrow = num_simulations, ncol = 30)
colnames(results_matrix) <- c("mvbcf_1k_pehe1", "mvbcf_1k_pehe2","mvbcf_0.5k_pehe1", "mvbcf_0.5k_pehe2","mvbcf_0.25k_pehe1", "mvbcf_0.25k_pehe2","mvbcf_0.1k_pehe1", "mvbcf_0.1k_pehe2",
                             "mvbcf_0.05k_pehe1", "mvbcf_0.05k_pehe2",
                             "mvbcf_1k_tau_951", "mvbcf_1k_tau_952","mvbcf_0.5k_tau_951", "mvbcf_0.5k_tau_952","mvbcf_0.25k_tau_951", "mvbcf_0.25k_tau_952","mvbcf_0.1k_tau_951", "mvbcf_0.1k_tau_952",
                             "mvbcf_0.05k_tau_951", "mvbcf_0.05k_tau_952", "mvbcf_1k_tau_951w", "mvbcf_1k_tau_952w","mvbcf_0.5k_tau_951w", "mvbcf_0.5k_tau_952w","mvbcf_0.25k_tau_951w", "mvbcf_0.25k_tau_952w","mvbcf_0.1k_tau_951w", "mvbcf_0.1k_tau_952w",
                             "mvbcf_0.05k_tau_951w", "mvbcf_0.05k_tau_952w")

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-runif(n)
X2<-runif(n)
X3<-runif(n)
X4<-runif(n)
X5<-runif(n)
X6<-rbinom(n, 1, 0.5)
X7<-rbinom(n, 1, 0.5)
X8<-rbinom(n, 1, 0.5)
X9<-sample(c(0, 1, 2, 3, 4), n, replace=T)
X10<-sample(c(0, 1, 2, 3, 4), n, replace=T)

X<-cbind(X1, X2, X3, X4, X5, X6, X7, X8, X9, X10)

Mu1<-(11*sin(pi*X4*X5)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
Mu2<-(9*sin(pi*X1*X2)+22*(X3-0.5)^2+14*X4+8*X6+X9)*10+300

Tau1<-(2*X4+2*X2)*10
Tau2<-(1*X3+3*X5)*10

true_propensity<-X4

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-runif(n_test)
X2_test<-runif(n_test)
X3_test<-runif(n_test)
X4_test<-runif(n_test)
X5_test<-runif(n_test)
X6_test<-rbinom(n_test, 1, 0.5)
X7_test<-rbinom(n_test, 1, 0.5)
X8_test<-rbinom(n_test, 1, 0.5)
X9_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)
X10_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)

X_test<-cbind(X1_test, X2_test, X3_test, X4_test, X5_test, X6_test, X7_test, X8_test, X9_test, X10_test)

Mu1_test<-(11*sin(pi*X4_test*X5_test)+18*(X3_test-0.5)^2+10*X4_test+12*X6_test+X9_test)*10+300
Mu2_test<-(9*sin(pi*X1_test*X2_test)+22*(X3_test-0.5)^2+14*X4_test+8*X6_test+X9_test)*10+300

Tau1_test<-(2*X4_test+2*X2_test)*10
Tau2_test<-(1*X3_test+3*X5_test)*10

true_propensity_test<-X4_test

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-50
n_burn<-0
num_gfr<-1000

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

mvbcf_1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_1k_tau_preds1<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_1k_ate1<-mean(mvbcf_1k_tau_preds1)
mvbcf_1k_tau_preds2<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_1k_ate2<-mean(mvbcf_1k_tau_preds2)

mvbcf_1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_1k_tau_preds1)^2))
mvbcf_1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_1k_tau_951<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_1k_tau_951w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_tau_952<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_1k_tau_952w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-500

mvbcf_0.5k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.5k_tau_preds1<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.5k_ate1<-mean(mvbcf_0.5k_tau_preds1)
mvbcf_0.5k_tau_preds2<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.5k_ate2<-mean(mvbcf_0.5k_tau_preds2)

mvbcf_0.5k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.5k_tau_preds1)^2))
mvbcf_0.5k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.5k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.5k_tau_951<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.5k_tau_951w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_tau_952<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.5k_tau_952w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-250

mvbcf_0.25k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.25k_tau_preds1<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.25k_ate1<-mean(mvbcf_0.25k_tau_preds1)
mvbcf_0.25k_tau_preds2<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.25k_ate2<-mean(mvbcf_0.25k_tau_preds2)

mvbcf_0.25k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.25k_tau_preds1)^2))
mvbcf_0.25k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.25k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.25k_tau_951<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.25k_tau_951w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_tau_952<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.25k_tau_952w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-100

mvbcf_0.1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.1k_tau_preds1<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.1k_ate1<-mean(mvbcf_0.1k_tau_preds1)
mvbcf_0.1k_tau_preds2<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.1k_ate2<-mean(mvbcf_0.1k_tau_preds2)

mvbcf_0.1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.1k_tau_preds1)^2))
mvbcf_0.1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.1k_tau_951<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.1k_tau_951w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_tau_952<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.1k_tau_952w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-50

mvbcf_0.05k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.05k_tau_preds1<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.05k_ate1<-mean(mvbcf_0.05k_tau_preds1)
mvbcf_0.05k_tau_preds2<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.05k_ate2<-mean(mvbcf_0.05k_tau_preds2)

mvbcf_0.05k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.05k_tau_preds1)^2))
mvbcf_0.05k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.05k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.05k_tau_951<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.05k_tau_951w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_tau_952<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.05k_tau_952w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


# Store the results in the matrix
  results_matrix[i, ] <- c(mvbcf_1k_pehe1, mvbcf_1k_pehe2, mvbcf_0.5k_pehe1, mvbcf_0.5k_pehe2, mvbcf_0.25k_pehe1, mvbcf_0.25k_pehe2, mvbcf_0.1k_pehe1, mvbcf_0.1k_pehe2,
                             mvbcf_0.05k_pehe1, mvbcf_0.05k_pehe2,
                             mvbcf_1k_tau_951, mvbcf_1k_tau_952, mvbcf_0.5k_tau_951, mvbcf_0.5k_tau_952, mvbcf_0.25k_tau_951, mvbcf_0.25k_tau_952, mvbcf_0.1k_tau_951, mvbcf_0.1k_tau_952,
                             mvbcf_0.05k_tau_951, mvbcf_0.05k_tau_952, mvbcf_1k_tau_951w, mvbcf_1k_tau_952w, mvbcf_0.5k_tau_951w, mvbcf_0.5k_tau_952w, mvbcf_0.25k_tau_951w, mvbcf_0.25k_tau_952w, mvbcf_0.1k_tau_951w, mvbcf_0.1k_tau_952w,
                             mvbcf_0.05k_tau_951w, mvbcf_0.05k_tau_952w)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "XMVBCF_simulation_results_DGP3.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to XMVBCF_simulation_results_DGP3.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 59381 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 32815 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14369 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6663 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4262 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 52492 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 27224 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14629 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6798 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4242 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


...

Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 52954 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 27146 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14454 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 7006 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4310 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 52685 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 27489 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13702 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6989 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4298 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 51691 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 27523 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14581 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6813 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4255 ms
Simulation completed and results saved to XMVBCF_simulation_results_DGP3.csv


1m and 7.5s per iteration

1 hour and 53 minutes per 100 replications

In [5]:
print(results_matrix)

       mvbcf_1k_pehe1 mvbcf_1k_pehe2 mvbcf_0.5k_pehe1 mvbcf_0.5k_pehe2
  [1,]      11.438697      11.448678        11.990835        14.842634
  [2,]      13.104853       9.159564        11.331811         7.488667
  [3,]      11.583048       7.692171        11.447170         9.800578
  [4,]      10.104844       9.723456         9.178929        11.693980
  [5,]       9.598064      11.374810        12.576672         7.522533
  [6,]       9.473005       8.385594        12.765600         8.640537
  [7,]       9.378642      10.679211         7.749549         9.398756
  [8,]       7.292608      15.320515         9.378857        12.925247
  [9,]      13.042647       7.161590        11.249505         8.656883
 [10,]      11.257352      14.588212        14.297096        11.944865
 [11,]      11.776619      12.965216        10.319010        14.949258
 [12,]       7.725602      11.011391         7.607435        11.208102
 [13,]       9.896094      16.296902         7.655501        13.595184
 [14,]